# *Data Loading*

In [2]:
import pandas as pd
import requests
from datetime import timedelta

df = pd.read_csv("../data/dataset_with_weather_feature.csv")
df.head()

,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delayed,...,holiday_or_weekend_transit_flag,holiday_proximity_feature,api_temperature,api_humidity,api_wind_speed,precipitation,bad_weather_flag_api,temperature,humidity,wind_speed
0,250.99,amazon logistics,automobile parts,ev bike,standard,west,clear,235.6,48.07,no,...,0,1,32.483571,74.097849,17.056317,3.506765,0,32.483571,74.097849,17.056317
1,250.99,amazon logistics,clothing,bike,express,central,clear,81.8,45.51,yes,...,0,1,29.308678,94.778295,15.486040,2.943648,0,29.308678,94.778295,15.486040
2,250.99,amazon logistics,clothing,van,same day,north,clear,282.9,31.33,yes,...,1,8,33.238443,56.395895,13.558508,1.892765,0,33.238443,56.395895,13.558508
3,250.99,amazon logistics,cosmetics,ev bike,two day,central,stormy,88.6,8.67,no,...,0,2,37.615149,88.800189,7.938182,2.332062,1,37.615149,88.800189,7.938182
4,250.99,amazon logistics,cosmetics,ev van,two day,east,foggy,204.2,8.09,no,...,1,1,28.829233,72.374135,2.726783,1.645159,1,28.829233,72.374135,2.726783


In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 44 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   delivery_id                      25000 non-null  float64
 1   delivery_partner                 25000 non-null  object 
 2   package_type                     25000 non-null  object 
 3   vehicle_type                     25000 non-null  object 
 4   delivery_mode                    25000 non-null  object 
 5   region                           25000 non-null  object 
 6   weather_condition                25000 non-null  object 
 7   distance_km                      25000 non-null  float64
 8   package_weight_kg                25000 non-null  float64
 9   delayed                          25000 non-null  object 
 10  delivery_status                  25000 non-null  object 
 11  delivery_rating                  25000 non-null  int64  
 12  delivery_cost     

,delivery_id,distance_km,package_weight_kg,delivery_rating,delivery_cost,expected_time_hours_recon,speed_kmph_recon,weather_mult_recon,delivery_time_hours_recon,partner_mult_recon,...,holiday_or_weekend_transit_flag,holiday_proximity_feature,api_temperature,api_humidity,api_wind_speed,precipitation,bad_weather_flag_api,temperature,humidity,wind_speed
count,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.0000,25000.000000,25000.000000,25000.000000,...,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,12500.500000,150.390436,25.145898,3.666000,864.944579,33.053344,39.9762,1.100310,5.360406,0.999994,...,0.621280,2.648600,30.004444,72.466140,10.014720,1.987841,0.299240,30.004444,72.466140,10.014720
std,7212.732314,86.409745,14.368663,1.149964,435.712593,17.592341,6.4653,0.064465,2.631007,0.003666,...,0.485078,2.627142,4.992622,12.993169,5.744667,1.985756,0.457934,4.992622,12.993169,5.744667
min,250.990000,3.600000,0.670000,1.000000,95.667400,8.000000,30.0000,1.000000,0.640518,0.993437,...,0.000000,0.000000,10.387999,50.000249,0.000937,0.000059,0.000000,10.387999,50.000249,0.000937
25%,6250.750000,75.900000,12.680000,3.000000,490.800000,24.000000,35.0000,1.050000,3.180450,0.998812,...,0.000000,1.000000,26.640340,61.202824,5.056816,0.561758,0.000000,26.640340,61.202824,5.056816
50%,12500.500000,151.000000,25.145000,4.000000,867.535000,26.400000,40.0000,1.100000,5.227587,1.000945,...,1.000000,2.000000,30.015309,72.493775,10.043579,1.389765,0.000000,30.015309,72.493775,10.043579
75%,18750.250000,224.900000,37.660000,5.000000,1237.910000,48.000000,45.0000,1.150000,7.307020,1.002187,...,1.000000,4.000000,33.370850,83.714342,15.016008,2.762550,1.000000,33.370850,83.714342,15.016008
max,24750.010000,297.100000,49.520000,5.000000,1632.720600,52.800000,50.0000,1.200000,13.682192,1.005584,...,1.000000,13.000000,52.395421,94.998747,19.998794,23.482742,1.000000,52.395421,94.998747,19.998794


In [4]:
df["order_ts_recon"] = pd.to_datetime(df["order_ts_recon"])
df["expected_ts_recon"] = pd.to_datetime(df["expected_ts_recon"])

df["order_date"] = df["order_ts_recon"].dt.date
df["expected_date"] = df["expected_ts_recon"].dt.date

In [5]:
years = df["order_ts_recon"].dt.year.unique()

print("Years in dataset:", years)

Years in dataset: [2024]


# *Noise Adding*

9% Gaussian Noise

In [9]:
import numpy as np

np.random.seed(42)

noise_level = 0.09

numeric_cols = [
    "distance_km",
    "package_weight_kg",
    "delivery_cost",
    "expected_time_hours_recon",
    "speed_kmph_recon",
    "api_temperature",
    "api_humidity",
    "api_wind_speed"
]

for col in numeric_cols:
    noise = np.random.normal(0, noise_level * df[col].std(), len(df))
    df[col] = df[col] + noise

df[numeric_cols] = df[numeric_cols].clip(lower=0)

In [12]:
df.to_csv("../data/dataset_with_weather_and_noise.csv", index=False)